# 8회차 실습 ② — MLflow로 앙상블까지 비교하기 (심화)

`train_optuna_ensemble.ipynb`를 참고해,
**로그 타깃 학습 + 단일 모델 + 앙상블(Voting/Stacking)** 을
MLflow로 비교하는 심화 실습입니다.

---

### 이 실습에서 하는 것

- **예측 타깃** : `daily_count` — 단, `log1p`로 변환해 학습하고 예측은 `expm1`로 되돌림
- **비교 대상** : 단일 모델(XGBoost · LightGBM) → Voting → Stacking
- **기록 지표** : RMSLE(기준) · RMSE · MAE
- **핵심 질문** : "앙상블이 단일 모델보다 나은가?" 를 MLflow UI에서 확인

---

> 💡 **실행 전 준비**
>
> ```bash
> pip install mlflow
> ```

## 1. 라이브러리 임포트

앙상블(Voting/Stacking)과 기본 모델(XGBoost/LightGBM/RandomForest),
그리고 MLflow를 불러옵니다.

In [1]:
import os
from datetime import datetime   # 실행 날짜 태그용
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

from sklearn.linear_model import Ridge   # Stacking의 최종(메타) 모델로 사용
from sklearn.ensemble import (
    RandomForestRegressor,
    VotingRegressor,     # 여러 모델 예측을 평균
    StackingRegressor,   # 여러 모델 예측을 메타 모델이 다시 학습
)
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import mlflow
import mlflow.sklearn

print("임포트 완료. MLflow 버전:", mlflow.__version__)

임포트 완료. MLflow 버전: 3.14.0


## 2. DB 연결 + 데이터 로드

실습①과 동일합니다. `.env`의 접속 정보로 MySQL에서 학습 데이터를 읽어옵니다.

In [2]:
load_dotenv()   # .env 로드

# DB 접속 정보 읽기
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = os.getenv("DB_PORT")
DB_NAME     = os.getenv("DB_NAME")

# 연결 문자열 조립 + 엔진 생성
DATABASE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

# 피처 + 타깃(daily_count)을 한 번에 뽑는 SQL
sql = """
    SELECT
        d.lot_id,                                   -- 주차장 ID
        i.capacity,                                 -- 총 주차면수
        MONTH(d.use_date)                       AS month,        -- 월(계절성)
        DAYOFWEEK(d.use_date)                   AS day_of_week,  -- 요일(1=일~7=토)
        CASE WHEN DAYOFWEEK(d.use_date) IN (1,7)
             THEN 1 ELSE 0 END                  AS is_weekend,   -- 주말 여부
        WEEKOFYEAR(d.use_date)                  AS week_of_year, -- 연중 주차
        COALESCE(h.holiday_date IS NOT NULL, 0) AS is_holiday,   -- 공휴일 여부
        d.use_date,                                 -- 날짜(정렬용)
        d.daily_count                           AS daily_count   -- ★ 타깃
    FROM parking_daily d
    INNER JOIN parking_info i ON d.lot_id   = i.id
    LEFT  JOIN holidays     h ON d.use_date = h.holiday_date
    WHERE d.daily_count > 0 AND i.capacity > 0
    ORDER BY d.use_date, d.lot_id
"""
df = pd.read_sql(text(sql), engine)   # 결과를 DataFrame으로
print(f"로드 완료: {len(df):,}행")

로드 완료: 21,869행


## 3. 피처/타깃 + 시계열 분할 + 로그 타깃

`daily_count`는 값이 크고 오른쪽으로 치우친 분포라 **`log1p` 변환**이 잘 맞습니다.
학습은 로그 값으로 하고, 예측은 나중에 `expm1`로 되돌립니다(4번 셀).

In [3]:
# 입력 피처(순서 고정) + 타깃
FEATURE_COLUMNS = [
    "lot_id", "capacity", "month", "day_of_week",
    "is_weekend", "week_of_year", "is_holiday",
]
TARGET_COLUMN = "daily_count"

# 시간순 정렬 후 8:2로 분할(셔플 금지)
df_sorted = df.sort_values("use_date").reset_index(drop=True)
X = df_sorted[FEATURE_COLUMNS]
y = df_sorted[TARGET_COLUMN]

split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# 학습에 쓸 '로그 타깃' (log1p = log(1+x), 0도 안전하게 처리)
y_train_log = np.log1p(y_train)

print(f"훈련셋: {len(X_train):,}행 / 테스트셋: {len(X_test):,}행")

훈련셋: 17,495행 / 테스트셋: 4,374행


## 4. 평가 지표 함수 (원본 스케일에서 계산)

모델은 로그 타깃으로 학습하므로 예측값도 로그 스케일입니다.
`expm1`(= exp(x)-1)로 **원래 대수 스케일로 되돌린 뒤** 지표를 계산합니다.

In [4]:
def rmsle(y, pred):
    # 로그 변환 후 RMSE(상대 오차)
    log_y    = np.log1p(y)
    log_pred = np.log1p(np.maximum(pred, 0))   # 음수 예측 방지
    return np.sqrt(np.mean((log_y - log_pred) ** 2))

def rmse(y, pred):
    return np.sqrt(mean_squared_error(y, pred))

def eval_metrics(y, pred):
    # 세 지표를 딕셔너리로 반환
    return {"rmsle": rmsle(y, pred), "rmse": rmse(y, pred), "mae": mean_absolute_error(y, pred)}

def predict_original(model, X):
    # 로그 타깃으로 학습한 모델의 예측을 원래(대수) 스케일로 되돌린다
    return np.expm1(model.predict(X))

## 5. MLflow 실험 설정

실습①과 **다른 실험 이름**을 씁니다. UI에서 두 실습이 섞이지 않고 따로 보입니다.

In [5]:
# 📌 방법 A — SQLite 백엔드 (MLflow 권장 방식)
#   파일 저장(file store) 대신 SQLite DB 파일에 실험을 기록한다.
#   - 현재 폴더에 mlflow.db 파일이 생기고 거기에 Run이 쌓인다.
#   - 파일 저장과 달리 최신 MLflow가 정식 지원하는 방식이다.
#   ⚠️ UI를 띄울 때 반드시 같은 백엔드를 지정해야 한다:
#       mlflow ui --backend-store-uri sqlite:///mlflow.db
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# 실험 그룹 지정(없으면 자동 생성)
mlflow.set_experiment("hangang_parking_ensemble_compare_v3")
print("실험 설정 완료:", mlflow.get_tracking_uri())

실험 설정 완료: sqlite:///mlflow.db


## 6. 공통 로깅 함수

모델 하나를 **학습 → 예측 → 평가 → MLflow 기록**까지 하는 과정을 함수로 묶습니다.
`log_target=True`면 로그 타깃으로 학습하고 `expm1`로 되돌려 평가합니다.
이렇게 묶어두면 단일 모델도 앙상블도 **같은 방식**으로 기록할 수 있습니다.

In [6]:
def train_and_log(name, model, log_target=True):
    # 하나의 Run으로 기록
    with mlflow.start_run(run_name=name):
        if log_target:
            model.fit(X_train, y_train_log)          # 로그 타깃으로 학습
            pred = predict_original(model, X_test)   # 예측 → expm1로 되돌림
        else:
            model.fit(X_train, y_train)              # (원값 학습 옵션)
            pred = model.predict(X_test)

        metrics = eval_metrics(y_test, pred)         # RMSLE/RMSE/MAE

        # MLflow 기록: 설정 + 성능 (모델 파일은 저장 안 함 - 비교용)
        mlflow.log_param("model", name)
        # 모델의 모든 하이퍼파라미터를 한 번에 기록(UI Parameters에 표시됨)
        mlflow.log_params(model.get_params())
        # 실행 날짜 태그(UI에서 tags.run_date로 날짜별 필터 가능)
        mlflow.set_tag("run_date", datetime.now().strftime("%Y-%m-%d %H:%M"))
        mlflow.log_param("log_target", log_target)
        mlflow.log_metric("rmsle", metrics["rmsle"])
        mlflow.log_metric("rmse",  metrics["rmse"])
        mlflow.log_metric("mae",   metrics["mae"])

        print(f"[{name}] RMSLE={metrics['rmsle']:.4f}  RMSE={metrics['rmse']:.1f}  MAE={metrics['mae']:.1f}")
        # 표 정리 + 최종 저장을 위해 모델 객체까지 함께 반환
        return {"name": name, **metrics, "model": model}

## 7. 단일 모델 학습·기록 (XGBoost, LightGBM)

먼저 비교의 기준이 될 단일 모델 두 개를 학습합니다.

In [7]:
results = []   # 모든 모델 결과를 모을 리스트

# 부스팅 계열 단일 모델 두 개
xgb  = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                    random_state=42, verbosity=0)
lgbm = LGBMRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                     random_state=42, verbose=-1)

# 학습 + MLflow 기록(공통 함수 사용)
results.append(train_and_log("XGBoost", xgb))
results.append(train_and_log("LightGBM", lgbm))

[XGBoost] RMSLE=0.7079  RMSE=957.6  MAE=654.5
[LightGBM] RMSLE=0.7067  RMSE=953.3  MAE=653.8


## 8. 앙상블 학습·기록 (Voting, Stacking)

- **Voting** : 여러 모델의 예측을 **평균** → 개별 모델의 실수를 서로 상쇄
- **Stacking** : 여러 모델의 예측을 입력으로 받는 **메타 모델**이 최종 예측

단일 모델과 똑같이 `train_and_log`로 기록해, 한 화면에서 비교합니다.

In [8]:
# 앙상블 내부에 넣을 기본 추정기들(단일 모델과 겹치지 않게 새 인스턴스 생성)
base_xgb  = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                         random_state=42, verbosity=0)
base_lgbm = LGBMRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                          random_state=42, verbose=-1)
base_rf   = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)

# Voting: 세 모델의 예측을 평균
voting = VotingRegressor(estimators=[
    ("xgb", base_xgb), ("lgbm", base_lgbm), ("rf", base_rf),
])
results.append(train_and_log("Voting", voting))

# Stacking: 세 모델의 예측을 Ridge(메타 모델)가 다시 학습해 최종 예측
stacking = StackingRegressor(
    estimators=[("xgb", base_xgb), ("lgbm", base_lgbm), ("rf", base_rf)],
    final_estimator=Ridge(alpha=1.0),
)
results.append(train_and_log("Stacking", stacking))

[Voting] RMSLE=0.7080  RMSE=958.2  MAE=650.8
[Stacking] RMSLE=0.6938  RMSE=974.2  MAE=668.6


## 9. 결과 비교 표

RMSLE 오름차순으로 정렬해 단일 모델과 앙상블을 한눈에 비교합니다.

In [9]:
# model 객체는 표에서 빼고 지표만 추려서 DataFrame으로
results_df = (
    pd.DataFrame([{k: v for k, v in r.items() if k != "model"} for r in results])
    .sort_values("rmsle").reset_index(drop=True)
)
results_df

,name,rmsle,rmse,mae
0,Stacking,0.693805,974.243317,668.574288
1,LightGBM,0.706662,953.302386,653.834415
2,XGBoost,0.707942,957.601300,654.523804
3,Voting,0.707967,958.218881,650.756538


## 10. 가장 좋은 모델을 pkl 번들로 저장 (배포용)

서빙 코드(`predict.py`)가 읽는 **번들(dict) 형식**으로 저장합니다.
`log_target`, `feature_columns` 등 메타 정보를 함께 담아야
예측 시 올바르게 역변환·정렬할 수 있습니다.

In [10]:
import joblib

# RMSLE가 가장 낮은(=1등) 모델 이름과 객체를 찾는다
best = results_df.iloc[0]["name"]
best_model = next(r["model"] for r in results if r["name"] == best)
print("최고 성능 모델:", best)

# 저장 폴더 준비(프로젝트 루트의 models_pkl)
output_dir = Path("..") / "models_pkl"
output_dir.mkdir(exist_ok=True)

# predict.py가 기대하는 번들 구조로 저장
bundle = {
    "model"          : best_model,       # 학습된 추정기
    "target"         : "daily_count",    # 무엇을 예측하는지
    "log_target"     : True,             # 로그 타깃 학습 → 예측 시 expm1 필요
    "feature_columns": FEATURE_COLUMNS,  # 입력 피처 순서
    "model_name"     : best,             # 참고용 이름
}
joblib.dump(bundle, output_dir / "hangang_parking.pkl")
print("저장 완료: models_pkl/hangang_parking.pkl")

최고 성능 모델: Stacking
저장 완료: models_pkl/hangang_parking.pkl


## 11. MLflow UI로 비교하기

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

> ⚠️ SQLite 백엔드는 UI를 띄울 때 **반드시 `--backend-store-uri`** 를 붙여야
> 노트북이 기록한 `mlflow.db`를 읽습니다. 그냥 `mlflow ui`만 실행하면
> 빈 화면(다른 저장소)이 보입니다.

```
1. http://127.0.0.1:5000 접속
2. 'hangang_parking_ensemble_compare' 실험 선택
3. XGBoost / LightGBM / Voting / Stacking Run을 함께 선택
4. [Compare] → RMSLE 기준으로 앙상블이 단일 모델보다 나은지 확인
```

---

### ✅ 정리

- 로그 타깃 학습 + 앙상블까지 **같은 함수**로 MLflow Run에 남겨 한 화면에서 비교했습니다.
- 지표는 **RMSLE / RMSE / MAE**, 예측은 `expm1`로 되돌려 원래 대수 스케일에서 평가합니다.
- 최고 모델을 `predict.py`가 읽는 **번들(dict)** 로 저장 → 그대로 배포할 수 있습니다.

In [11]:
# 기존에 떠 있는 MLflow UI를 종료, 터미널에서
# lsof -ti:5000 | xargs kill -9